# Ohio Voter Registration Analysis

**Workflow:** Download check → Load → Clean → Enrich → Explore → Export JSON + Excel

Run cells top-to-bottom on first use. After the initial load, you can re-run any individual cell without restarting — the DataFrame stays in memory between cells.

| Cell | Purpose | Re-run cost |
|---|---|---|
| 1 — Imports | Load libraries and logger | Instant |
| 2 — Config | Set county, paths | Instant |
| 3 — Load | Read SWVF files from disk | 10–30 s (county) / 2–4 min (Ohio) |
| 4 — Clean | Parse dates, normalise fields | 5–15 s |
| 5 — Participation | Compute per-voter metrics | 10–30 s |
| 6 — Explore | Ad-hoc queries (scratch space) | Instant |
| 7 — Export JSON | Write web dashboard data | 5–10 s |
| 8 — Export Excel | Write .xlsx workbook | 30–120 s |

In [ ]:
# ─── Cell 1: Imports and logging setup ───────────────────────────────────────
#
# Run once per session. If you edit voter_data_cleaner_v2.py, restart the
# kernel (Kernel → Restart) and re-run from this cell — Python's import
# cache won't pick up changes otherwise.
#
# Re-running this cell without restarting is safe — setup_logging() detects
# that the logger is already configured and reuses it without creating a new
# log file.

import sys
from pathlib import Path
from datetime import date

import polars as pl
import pandas as pd

# Import all analysis functions from the main script.
# voter_data_cleaner_v2.py must be in the same folder as this notebook.
from voter_data_cleaner_v2 import (
    setup_logging,
    _timer,
    load_voter_files,
    clean_voter_data,
    identify_election_cols,
    add_voter_participation,
    build_decade_summary,
    build_election_participation,
    build_district_breakdown,
    build_party_crosstabs,
    build_precinct_summary,
    build_county_summary,
    export_json,
    build_workbook,
    OHIO_COUNTIES,
)

log = setup_logging('notebook')
log.info('─' * 50)
log.info('Polars version : %s', pl.__version__)
log.info('Pandas version : %s', pd.__version__)
log.info('Python         : %s', sys.version.split()[0])
log.info('─' * 50)

In [ ]:
# ─── Cell 2: Configuration ───────────────────────────────────────────────────
#
# Change COUNTY_NUMBER to analyse a different county.
# Set COUNTY_NUMBER = None to load all 88 Ohio counties (full-state run).
#
# Ohio county reference (commonly used):
#   '57' = Montgomery (Dayton)   '25' = Franklin (Columbus)
#   '18' = Cuyahoga (Cleveland)  '31' = Hamilton (Cincinnati)
#   '77' = Summit (Akron)        '76' = Stark (Canton)

# Path() resolves to the directory the notebook is opened from, which is the
# project root when opened via VSCode. This avoids hardcoding a machine-specific
# path so the notebook works for any collaborator who clones the repo.
BASE_DIR      = Path().resolve()
TXT_DIR       = BASE_DIR / "source" / "State Voter Files"
TXT_FILES     = sorted(TXT_DIR.glob("SWVF_*.txt"))

COUNTY_NUMBER = '57'    # ← change this; or None for all of Ohio
COUNTY_NAME   = OHIO_COUNTIES.get(COUNTY_NUMBER, f'County {COUNTY_NUMBER}') \
                if COUNTY_NUMBER else 'Ohio (Statewide)'

log.info('County        : %s (%s)', COUNTY_NAME, COUNTY_NUMBER or 'all')
log.info('Voter files   : %d found', len(TXT_FILES))
for f in TXT_FILES:
    log.info('  %-25s  %.2f GB', f.name, f.stat().st_size / 1e9)

if not TXT_FILES:
    log.error('No SWVF_*.txt files found in %s', TXT_DIR)
    log.error('Run ohio_voter_pipeline.py first to download and decompress the voter files.')

In [ ]:
# ─── Cell 3: Load voter files ─────────────────────────────────────────────────
#
# Polars scans the files lazily and pushes the county filter into the reader,
# so only matching rows are decoded from disk — the full 7.9 GB is never loaded
# when analysing a single county.
#
# Expected time:
#   Single county  → 10–30 seconds
#   All of Ohio    → 2–4 minutes

with _timer(log, 'load voter files'):
    df_raw = load_voter_files(
        txt_files     = TXT_FILES,
        county_number = COUNTY_NUMBER,
        logger        = log,
    )

# Preview the raw schema — inspect column names and types before cleaning
print(f"\nRows: {len(df_raw):,}   Columns: {len(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# ─── Cell 4: Clean and enrich ─────────────────────────────────────────────────
#
# Parses DATE_OF_BIRTH and REGISTRATION_DATE into native date types,
# derives birth decade cohorts, normalises PARTY_AFFILIATION to REP/DEM/UNC,
# and drops records with unparseable or out-of-range birth years.
#
# Safe to re-run against df_raw without reloading from disk.

with _timer(log, 'clean voter data'):
    df = clean_voter_data(df_raw, log)

# Spot-check: party distribution after normalisation
print(f"\nRecords after cleaning: {len(df):,}")
df.group_by('PARTY_LABEL').len().sort('len', descending=True)

In [ ]:
# ─── Cell 5: Participation metrics ───────────────────────────────────────────
#
# Computes four new columns per voter:
#   Elections_Eligible — elections held after the voter's registration date
#   Elections_Voted    — of those, elections where they participated
#   Turnout_Rate       — ratio (null if 0 eligible)
#   Voter_Frequency    — Frequent / Moderate / Infrequent / No Eligible Elections
#
# Uses Polars sum_horizontal across all 89 election columns in one pass.

election_cols = identify_election_cols(df)
log.info('%d election columns detected: %s → %s',
         len(election_cols), election_cols[0], election_cols[-1])

with _timer(log, 'compute participation metrics'):
    df = add_voter_participation(df, election_cols, log)

# Frequency distribution sanity check
df.group_by('Voter_Frequency').len().sort('len', descending=True)

In [ ]:
# ─── Cell 6: Explore ─────────────────────────────────────────────────────────
#
# Scratch space — edit and re-run freely without affecting the DataFrame.
# The examples below can be replaced with any Polars query.
#
# Polars cheat sheet for common operations:
#   Filter rows      df.filter(pl.col('VOTER_STATUS') == 'ACTIVE')
#   Select columns   df.select(['LAST_NAME', 'FIRST_NAME', 'PARTY_LABEL'])
#   Group + count    df.group_by('X').len().sort('len', descending=True)
#   Cross-tab        df.pivot(on='PARTY_LABEL', index='Decade', values='n', aggregate_function='sum')
#   Filter + show    df.filter(pl.col('Turnout_Rate') == 1.0).head(10)

# Example 1: Top 10 precincts by total registered voters
(
    df.group_by('PRECINCT_NAME')
      .agg([
          pl.len().alias('Total'),
          pl.col('PARTY_LABEL').eq('REP').sum().alias('REP'),
          pl.col('PARTY_LABEL').eq('DEM').sum().alias('DEM'),
          pl.col('PARTY_LABEL').eq('UNC').sum().alias('UNC'),
          pl.col('Voter_Frequency').eq('Frequent (≥75%)').sum().alias('Frequent'),
      ])
      .sort('Total', descending=True)
      .head(10)
)

In [ ]:
# ─── Cell 6b: Explore — voter status breakdown ────────────────────────────────

# Example 2: Active vs. Confirmation by Congressional district
(
    df.group_by(['CONGRESSIONAL_DISTRICT', 'VOTER_STATUS'])
      .agg(pl.len().alias('Count'))
      .pivot(on='VOTER_STATUS', index='CONGRESSIONAL_DISTRICT', values='Count', aggregate_function='sum')
      .sort('CONGRESSIONAL_DISTRICT')
)

In [ ]:
# ─── Cell 6c: Explore — recent election turnout ───────────────────────────────

# Example 3: Turnout in the most recent General election by party
recent_general = [c for c in election_cols if c.startswith('GENERAL-')][-1]
log.info('Most recent General election column: %s', recent_general)

(
    df.with_columns(
        pl.col(recent_general).is_not_null()
          .and_(pl.col(recent_general).str.strip_chars() != '')
          .alias('voted')
    )
    .group_by('PARTY_LABEL')
    .agg([
        pl.len().alias('Total Registered'),
        pl.col('voted').sum().alias('Voted'),
    ])
    .with_columns(
        (pl.col('Voted') / pl.col('Total Registered') * 100).round(1).alias('Turnout %')
    )
    .sort('Turnout %', descending=True)
)

In [ ]:
# ─── Cell 7: Export JSON (web dashboard) ─────────────────────────────────────
#
# Writes four JSON files into docs/data/ in the exact schema that
# index.html / charts.js expects.  Replaces the sample data files.
# Also updates docs/manifest.json so the county appears in the dropdown
# and the 'Sample data' banner is cleared.
#
# After this cell completes, open docs/index.html in a browser (or use
# VSCode's Live Server extension) to see the live dashboard with real data.

with _timer(log, f'JSON export — {COUNTY_NAME}'):
    export_json(COUNTY_NAME, df, election_cols, log)

print(f"\nDashboard data written to: {BASE_DIR / 'docs' / 'data'}")
print("Open docs/index.html to view.")

In [ ]:
# ─── Cell 8: Export Excel workbook ───────────────────────────────────────────
#
# Writes a multi-sheet .xlsx workbook.
#
# include_raw=True  → includes a Voter Data sheet with all records
#                     (appropriate for county-scale; ~300K rows is fine in Excel)
# include_raw=False → replaces Voter Data with a County Summary sheet
#                     (required for Ohio-wide; 7.9M rows exceeds Excel's limit)
#
# The same summary data is in both the Excel workbook and the JSON files —
# they are computed from the same in-memory DataFrame.

include_raw = COUNTY_NUMBER is not None   # True for county, False for Ohio-wide

if COUNTY_NUMBER:
    output_path = BASE_DIR / f"county_{COUNTY_NUMBER}_analysis_{date.today()}.xlsx"
else:
    output_path = BASE_DIR / f"ohio_analysis_{date.today()}.xlsx"

with _timer(log, 'write Excel workbook'):
    build_workbook(
        df            = df,
        election_cols = election_cols,
        output_path   = output_path,
        county_name   = COUNTY_NAME,
        include_raw   = include_raw,
        logger        = log,
    )

print(f"\nWorkbook saved: {output_path}")
print(f"Size: {output_path.stat().st_size / 1e6:.1f} MB")

---
## Reference

### Switching counties
Change `COUNTY_NUMBER` in Cell 2, then re-run Cells 2 → 3 → 4 → 5 → 7 → 8.
You do not need to restart the kernel.

### Running for all of Ohio
Set `COUNTY_NUMBER = None` in Cell 2, then re-run from Cell 2.
Cell 8 will automatically switch to `include_raw=False` (County Summary sheet instead of raw Voter Data).

### Downloading updated voter files
Run `python ohio_voter_pipeline.py` from the VSCode terminal, then re-run from Cell 3.

### Log files
Each session writes a timestamped log to `logs/voter_analysis_YYYYMMDD_HHMMSS_notebook.log`.
Open it for full DEBUG-level output including timings and any warnings.
Re-running Cell 1 without restarting the kernel reuses the existing log file.

### Ohio county numbers (quick reference)

| # | County | # | County | # | County |
|---|---|---|---|---|---|
| 18 | Cuyahoga (Cleveland) | 25 | Franklin (Columbus) | 31 | Hamilton (Cincinnati) |
| 57 | Montgomery (Dayton) | 76 | Stark (Canton) | 77 | Summit (Akron) |
| 47 | Lorain | 48 | Lucas (Toledo) | 78 | Trumbull (Warren) |